In [ ]:
import numpy as np
import xarray as xr
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
def ax_pos_inch_to_absolute(fig_size, ax_pos_inch):
    ax_pos_absolute = []
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])

    return ax_pos_absolute

In [ ]:
def ax_pos_cm_to_absolute(fig_size, ax_pos_cm):
    ax_pos_absolute = []
    ax_pos_inch = [ pos / 2.54 for pos in ax_pos_cm ]
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])
    
    return ax_pos_absolute

In [ ]:
def angle_to_l(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 180. / x[~near_zero]
    return x


In [ ]:
def freq_to_time(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 1. / x[~near_zero]
    return x


In [ ]:
# grid
ntrunc=72
nSampWin = 720
spd = 2
nWindow = 59
frequency = np.fft.fftfreq(nSampWin, 1./spd)
ff = np.fft.fftshift(frequency[:])
ll = np.arange(0, ntrunc+1, 1)

In [ ]:
# base dir
base_dir = (Path.cwd() / "../../").resolve()
data_dir = base_dir / "data"
save_dir = base_dir / "figures"

In [ ]:
file_name = "ou-realization-2024-space-time-analysis-window-360-skip-180-epsilon0-5.8-lambda0-0.06-tau0-2.3.nc"
ds_ou = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
Fwflm_ou = ds_ou.Fwflm_real.values[:, :, :, :] + 1j * ds_ou.Fwflm_imag.values[:, :, :, :]
Fwtlm_ou = ds_ou.Fwtlm_real.values[:, :, :, :] + 1j * ds_ou.Fwtlm_imag.values[:, :, :, :]

In [ ]:
Fwflm_ou = np.fft.fftshift(Fwflm_ou, axes=1)

In [ ]:
# angular variance -- ou
Cl_ou = np.zeros(ntrunc+1)

for l in range(ntrunc+1):
    Cl_ou[l] = np.mean(np.mean(np.mean(np.abs(Fwtlm_ou[:, :, l, ntrunc-l:ntrunc+l+1])**2, axis=0), axis=0), axis=0)

# compensating for hanning window
Cl_ou *= 8. / 3.


In [ ]:
# temporal psd -- ou
Cfl_ou = np.zeros((nSampWin, ntrunc+1))

for l in range(ntrunc+1):
    Cfl_ou[:, l] = np.mean(np.mean(np.abs(Fwflm_ou[:, :, l, ntrunc-l:ntrunc+l+1])**2, axis=0), axis=1)

Cf_ou = np.mean(Cfl_ou, axis=1)

# compensating for hanning window
Cf_ou *= 8. / 3.

# for psd need to divide by dw = frequency resolution = sample rate / number of samples
Cf_ou *= nSampWin / 2.


In [ ]:
# because the variance changes by 2 orders of magnitudes it's better to fit the scaled data

In [ ]:
# cmb scaling
for l in range(ntrunc+1):
    Cl_ou[l] = l*(l+1) / (2*np.pi) * Cl_ou[l]

In [ ]:
# fit angular variance
# a**2 / (1 + b**2 * l(l+1))**2 * l*(l+1) / 2pi

popt_l, pcov_l = curve_fit(lambda l, a, b: a**2 * l * (l+1) / (2*np.pi) / ( 1 + b**2 * l * (l+1) )**2, ll[2:], Cl_ou[2:], p0=[5.5, 0.05])
perr_l = np.sqrt(np.diag(pcov_l))

Cl_ebcm = popt_l[0]**2 * ll * (ll+1) / (2*np.pi) / ( 1 + popt_l[1]**2 * ll * (ll+1) )**2

print(popt_l[0], perr_l[0])
print(popt_l[1], perr_l[1])

In [ ]:
# fit temporal
# a**2 * b / (1 + b**2 * f**2)**2

# need to take the mean over l of the formula in the text 
def temporal_variance(f, epsilon_0, lambda_0, tau_0):
    temporal_variance_l = np.zeros((f.shape[0], ntrunc+1))
    for l in range(0, ntrunc+1):
        tau_l = tau_0 / (1 + lambda_0**2 * l * (l+1))
        epsilon_l = epsilon_0 * tau_l / tau_0
        temporal_variance_l[:, l] =  2 * epsilon_l**2 * tau_l / (1 + tau_l**2 * f**2 * 4 * np.pi**2)
    return np.mean(temporal_variance_l, axis=1)

# fit only on subannual timescales
ind = np.where((ff > 1./360) & (ff < 1))[0]

popt_w, pcov_w = curve_fit(lambda f, c: temporal_variance(f, epsilon_0=popt_l[0], lambda_0=popt_l[1], tau_0=c), ff[ind], Cf_ou[ind], p0=[2.5])
perr_w = np.sqrt(np.diag(pcov_w))

# plot on all scales
ind_fit_w = np.where((ff >= 0) & (ff <= 1))[0]

Cf_ebcm = temporal_variance(ff[ind_fit_w], popt_l[0], popt_l[1], popt_w[0])

print(popt_w[0], perr_w[0])
# print(popt_w[1], perr_w[1])
# print(popt_w[2], perr_w[2])

In [ ]:
fig_size = (14.00/2.54, 17.00/2.54)
fig = plt.figure(figsize=fig_size)

ax = []

ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.75, 10.00, 12.00, 06.00])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.75, 01.50, 12.00, 06.00])))


##### ax0 #####

ax[0].plot(ll, Cl_ou, linestyle='', marker='.', color='C1', label='Ornstein–Uhlenbeck', alpha=1)

# standard error in the variance over windows and over m 
ax[0].fill_between(ll, (Cl_ou + Cl_ou*(2/(2*ll+1)/58)**0.5), (Cl_ou - Cl_ou*(2/(2*ll+1)/58)**0.5), color='C1', alpha=0.2) #, label='Standard\nerror')

# ebcm
ax[0].plot(ll, Cl_ebcm, linestyle='-', linewidth=1, marker='', color='black', label=r'$\mathcal{E}\,$BCM')

secax = ax[0].secondary_xaxis('top', functions=(angle_to_l, angle_to_l))
secax.set_xlabel('Angular scale')
# secax.set_xticks(np.array([90, 20, 10, 7, 6, 5, 4, 3]))
secax.set_xticks(np.array([18, 9, 6, 4.5, 3.6, 3]))
secax.xaxis.set_major_formatter(StrMethodFormatter(u"{x:.1f}°"))
secax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')

ax[0].legend(ncol=1, fontsize=10, frameon=True, edgecolor='grey')
# ax[0].set_title('Angular power spectrum', fontweight='bold')
ax[0].set_xlabel(r'Multipole moment ($\ell$)', fontsize=10)
ax[0].set_ylabel(r'$\ell (\ell+1) \, C_{\ell} \, / \, 2\pi$  $\mathrm{ [W \, m^{-2}]^{2}}$', fontsize=10)
ax[0].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
# ax[0].set_xscale('log')
# ax[0].set_yscale('log')
# ax[0].set_xlim(1, 80)
ax[0].set_ylim(0, 450)

# ax[0].grid()

ax[0].text(-00.05, 01.05, 'A', ha='right', va='bottom', transform=ax[0].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax1 #####

ax[1].plot(ff, Cf_ou, linestyle='-', marker='', color='C1', alpha=1)

# standard error in the variance over windows and over m
ax[1].fill_between(ff, Cf_ou*(10**(0.434*Cf_ou*(2/(2*0+1)/58)**0.5/Cf_ou)), Cf_ou/(10**(0.434*Cf_ou*(2/(2*0+1)/58)**0.5/Cf_ou)), color='C1', alpha=0.2)

# ebcm
ax[1].plot(ff[ind_fit_w], Cf_ebcm, linestyle='-', linewidth=1, marker='', color='black')

secax = ax[1].secondary_xaxis('top', functions=(freq_to_time, freq_to_time))
secax.set_xlabel('Period [day]', fontsize=10)
# secax.set_xticks(np.array([90, 0, 60, 30, 10, 5, 2]))
secax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secax.tick_params(axis='both', which='minor', direction='out', size=2, color='0.8')

# ax[0].legend(ncol=1)
# ax[0].set_title('Temporal power spectrum', fontweight='bold')
ax[1].set_xlabel(r'Frequency ($\omega$) [cpd]', fontsize=10)
ax[1].set_ylabel(r'$\hat{C}_{0}$ $\mathrm{ [W \, m^{-2}]^{2} \, day}$', fontsize=10)
ax[1].yaxis.set_label_coords(-0.09, 0.5)
ax[1].set_xscale('log')
ax[1].set_yscale('log')
ax[1].set_xlim(1./360, None)
ax[1].set_ylim(1e-1, 1e2)
ax[1].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[1].tick_params(axis='both', which='minor', direction='out', size=2, color='0.8')

ax[1].text(-00.05, 01.05, 'B', ha='right', va='bottom', transform=ax[1].transAxes, fontsize=10, fontweight='bold', color='black')


In [ ]:
file_name = "fig-s01"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')